<a href="https://colab.research.google.com/github/Sahanamurnal1108/Medical-summary-generation/blob/main/Copy_of_RAG_Test_copy_of_original_of_LLaMA_3_1_rag_validation_run18_05_02026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers accelerate bitsandbytes sentencepiece
!pip install -q huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.2 MB/s eta 0:00:00


In [ ]:
from huggingface_hub import login
login()

In [ ]:
all_json_outputs = []

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "meta-llama/Llama-3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("✅ LLaMA 3.1 loaded successfully!")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

✅ LLaMA 3.1 loaded successfully!


In [ ]:
from pydantic import BaseModel, ValidationError

class DischargeData(BaseModel):
    patient_name: str
    uhid: str
    age: int
    gender: str
    admission_date: str
    discharge_date: str
    department: str
    primary_diagnosis: str
    secondary_diagnosis: list
    complaints: list
    vitals: dict
    investigations: list
    treatment: list
    discharge_medications: list
    instructions: list
    follow_up_date: str

def validate_json(json_text):
    try:
        data = json.loads(json_text)
        validated = DischargeData(**data)
        return validated
    except Exception as e:
        print("Validation error:", e)
        return None

In [ ]:
import json
import re
def extract_json(transcript):
   prompt = f""" You are a medical information extraction system.
   Extract structured data from the transcript below.
   STRICT RULES:
   - For 'patient_name', prioritize extracting a full name. If a full name is not present, extract a descriptive identifier (e.g., 'X-year-old Y gender patient'). Only use 'null' if no patient identifier can be found at all.
   - If a specific field's value cannot be found in the transcript, use 'null' for string values and '0' for numeric values (like age, HR, Temp).
   - Output ONLY valid JSON.
   - Do not explain anything.
   - Do not add extra text.
   - Wrap the JSON output in ```json and ``` markers.
   JSON FORMAT: You MUST extract the information for each field from the provided transcript.
   ```json
   {{ "patient_name": "<EXTRACT PATIENT NAME>",
   "uhid": "<EXTRACT UHID>",
    "age": <EXTRACT AGE>,
    "gender": "<EXTRACT GENDER>",
    "admission_date": "<EXTRACT ADMISSION DATE>",
    "discharge_date": "<EXTRACT DISCHARGE DATE>",
     "department": "<EXTRACT DEPARTMENT>",
     "primary_diagnosis": "<EXTRACT PRIMARY DIAGNOSIS>",
      "secondary_diagnosis": [<EXTRACT SECONDARY DIAGNOSES>],
      "complaints": [<EXTRACT COMPLAINTS>],
      "vitals": {{ "BP": "<EXTRACT BLOOD PRESSURE>", "HR": <EXTRACT HEART RATE>, "Temp": <EXTRACT TEMPERATURE>, "Height": "<EXTRACT HEIGHT>", "Weight": "<EXTRACT WEIGHT>" }},
      "investigations": [<EXTRACT INVESTIGATIONS>],
      "treatment": [<EXTRACT TREATMENT>],
      "discharge_medications": [<EXTRACT DISCHARGE MEDICATIONS>],
      "instructions": [<EXTRACT INSTRUCTIONS>],
      "follow_up_date": "<EXTRACT FOLLOW UP DATE>"
       }}
   ```
       Transcript:
       {transcript}
       """
   inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
   output = model.generate( **inputs, max_new_tokens=700, temperature=0.1 ) # Reduced max_new_tokens
   # Decode only the newly generated tokens, excluding the input prompt
   result = tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
   print("Raw Model Output:", result) # Added print statement for debugging

   # First, try to extract JSON from a ```json code block
   json_code_block_match = re.search(r'```json\s*(.*?)\s*```', result, re.DOTALL)
   if json_code_block_match:
       json_str = json_code_block_match.group(1)
       try:
           json.loads(json_str)
           return json_str
       except json.JSONDecodeError:
           pass # If parsing fails, fall through to the next method

   # Fallback: Find all JSON-like objects in the result using a non-greedy match
   json_candidates = re.findall(r'\{.*?\}', result, re.DOTALL)

   # Iterate backwards to find the last valid JSON object
   for candidate in reversed(json_candidates):
       try:
           json.loads(candidate)
           return candidate # Return the first valid JSON found from the end
       except json.JSONDecodeError:
           continue
   return None # No valid JSON found

In [ ]:
def generate_summary(structured_json):

    prompt = f"""
Generate a professional discharge summary using the structured data below:

{structured_json}

Format it clearly like a hospital discharge document.
"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    output = model.generate(
        **inputs,
        max_new_tokens=800,
        temperature=0.3
    )

    return tokenizer.decode(output[0], skip_special_tokens=True)

In [ ]:
import sys
!{sys.executable} -m pip install reportlab
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.pagesizes import A4

def export_pdf(text, filename="discharge_summary.pdf"):

    doc = SimpleDocTemplate(filename, pagesize=A4)
    styles = getSampleStyleSheet()
    elements = []

    for line in text.split("\n"):
        elements.append(Paragraph(line, styles["Normal"]))
        elements.append(Spacer(1, 6))

    doc.build(elements)
    print("PDF Generated Successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 27.3 MB/s eta 0:00:00


In [ ]:
# List of transcripts
transcripts = [
    """ Abnormal serum PSA of 16 ng/ml, dribbling urine, inability to empty bladder, nocturia, urinary hesitancy and slow urine stream.", Urology, Elevated PSA - H&P ,"CHIEF COMPLAINT:, This 61-year-old male presents today with recent finding of abnormal serum PSA of 16 ng/ml. Associated signs and symptoms: Associated signs and symptoms include dribbling urine, inability to empty bladder, nocturia, urinary hesitancy and urine stream is slow.  Timing (onset/frequency): Onset was 6 months ago. Patient denies fever and chills and denies flank pain.,ALLERGIES: ,Patient admits allergies to adhesive tape resulting in severe rash. Patient denies an allergy to anesthesia.,MEDICATION HISTORY:, Patient is not currently taking any medications.,PAST MEDICAL HISTORY:, Childhood Illnesses: (+) asthma, Cardiovascular Hx: (-) angina, Renal / Urinary Hx: (-) kidney problems.,PAST SURGICAL HISTORY:, Patient admits past surgical history of appendectomy in 1992.,SOCIAL HISTORY:, Patient admits alcohol use, Drinking is described as heavy, Patient denies illegal drug use, Patient denies STD history, Patient denies tobacco use.,FAMILY HISTORY:, Patient admits a family history of gout attacks associated with father.,REVIEW OF SYSTEMS:, Unremarkable with exception of chief complaint.,PHYSICAL EXAM: ,BP Sitting: 120/80 Resp: 20 HR: 72 Temp: 98.6,The patient is a pleasant, 61-year-old male in no apparent distress who looks his given age, is well-developed and nourished with good attention to hygiene and body habitus.,Neck: Neck is normal and symmetrical, without swelling or tenderness. Thyroid is smooth and symmetric with no enlargement, tenderness or masses noted.,Respiratory: Respirations are even without use of accessory muscles and no intercostal retractions noted. Breathing is not labored, diaphragmatic, or abdominal. Lungs clear to auscultation with no rales, rhonchi, wheezes, or rubs noted.,Cardiovascular: Normal S1 and S2 without murmurs, gallop, rubs or clicks.  Peripheral pulses full to palpation, no varicosities, extremities warm with no edema or tenderness.,Gastrointestinal: Abdominal organs, bladder, kidney: No abnormalities, without masses, tenderness, or rigidity. Hernia: absent; no inguinal, femoral, or ventral hernias noted. Liver and/or Spleen: no abnormalities, tenderness, or masses noted. Stool specimen not indicated.,Genitourinary: Anus and perineum: no abnormalities. No fissures, edema, dimples, or tenderness noted.,Scrotum: no abnormalities. No lesions, rash, or sebaceous cyst noted.,Epididymides: no abnormalities, masses, or spermatocele, without enlargement, induration, or tenderness.,Testes: symmetrical; no abnormalities, tenderness, hydrocele, or masses noted.,Urethral Meatus: no abnormalities; no hypospadias, lesions, polyps, or discharge noted.,Penis: no abnormalities; circumcised; no phimosis, Peyronie's, condylomata, or lumps noted.,Prostate: size 60 gr, RT>LT and firm.,Seminal Vesicles: no abnormalities; symmetrical; no tenderness, induration, or nodules noted.,Sphincter tone: no abnormalities; good tone; without hemorrhoids or masses.,Skin/Extremities: Skin is warm and dry with normal turgor and there is no icterus. No skin rash, subcutaneous nodules, lesions or ulcers observed.,Neurological/Psychiatric: Oriented to person, place and time. Mood and affect normal, appropriate to situation, without depression, anxiety, or agitation.,TEST RESULTS:, No tests to report at this time.,IMPRESSION: ,Elevated prostate specific antigen (PSA).,PLAN:, Cystoscopy in the office.,DIAGNOSTIC & LAB ORDERS:, Ordered serum creatinine. Urinalysis and C & S ordered using clean-catch specimen. Ordered free prostate specific antigen (PSA). Ordered ultrasound of prostate.,I have discussed the findings of this follow-up evaluation with the patient. The discussion included a complete verbal explanation of any changes in the examination results, diagnosis and current treatment plan. Discussed the possibility of a TURP surgical procedure; risks, complications, benefits, and alternative measures discussed. There are no activity restrictions . Instructed Ben to avoid caffeinated or alcoholic beverages and excessively spiced foods. Questions answered. If any questions should arise after returning home I have encouraged the patient to feel free to call the office at 327-8850.,PRESCRIPTIONS: , Proscar Dosage: 5 mg tablet Sig: once daily Dispense: 30 Refills: 0 Allow Generic: No,PATIENT INSTRUCTIONS:,  Patient completed benign prostatic hypertrophy questionnaire.",

        """,
        """Patient with a history of gross hematuria.  CT scan was performed, which demonstrated no hydronephrosis or upper tract process; however, there was significant thickening of the left and posterior bladder wall.", Urology, Bladder Cancer ,"CHIEF COMPLAINT: , Bladder cancer.,HISTORY OF PRESENT ILLNESS:,  The patient is a 68-year-old Caucasian male with a history of gross hematuria.  The patient presented to the emergency room near his hometown on 12/24/2007 for evaluation of this gross hematuria.  CT scan was performed, which demonstrated no hydronephrosis or upper tract process; however, there was significant thickening of the left and posterior bladder wall.  Urology referral was initiated and the patient was sent to be evaluated by Dr. X. He eventually underwent a bladder biopsy on 01/18/08, which demonstrated high-grade transitional cell carcinoma without any muscularis propria in the specimen.  Additionally, the patient underwent workup for a right adrenal lesion, which was noted on the initial CT scan.  This workup involved serum cortisol analysis as well as potassium and aldosterone and ACTH level measurement.  All of this workup was found to be grossly negative.  Secondary to the absence of muscle in the specimen, the patient was taken back to the operating room on 02/27/08 by Dr. X and the tumor was noted to be very large with significant tumor burden as well as possible involvement of the bladder neck.  At that time, the referring urologist determined the tumor to be too large and risky for local resection, and the patient was referred to ABCD Urology for management and diagnosis.  The patient presents today for evaluation by Dr. Y.,PAST MEDICAL HISTORY: , Includes condyloma, hypertension, diabetes mellitus, hyperlipidemia, undiagnosed COPD, peripheral vascular disease, and claudication.  The patient denies coronary artery disease.,PAST SURGICAL HISTORY:,  Includes bladder biopsy on 01/18/08 without muscularis propria in the high-grade TCC specimen and a gun shot wound in 1984 followed by exploratory laparotomy x2.  The patient denies any bowel resection or GU injury at that time; however, he is unsure.,CURRENT MEDICATIONS:,1.  Metoprolol 100 mg b.i.d.,2.  Diltiazem 120 mg daily.,3.  Hydrocodone 10/500 mg p.r.n.,4.  Pravastatin 40 mg daily.,5.  Lisinopril 20 mg daily.,6.  Hydrochlorothiazide 25 mg daily.,FAMILY HISTORY: , Negative for any GU cancer, stones or other complaints.  The patient states he has one uncle who died of lung cancer.  He denies any other family history.,SOCIAL HISTORY: , The patient smokes approximately 2 packs per day times greater than 40 years.  He does drink occasional alcohol approximately 5 to 6 alcoholic drinks per month.  He denies any drug use.  He is a retired liquor store owner.,PHYSICAL EXAMINATION:,GENERAL:  He is a well-developed, well-nourished Caucasian male, who appears slightly older than stated age.  VITAL SIGNS:  Temperature is 96.7, blood pressure is 108/57, pulse is 75, and weight of 193.8 pounds.  HEAD AND NECK:  Normocephalic atraumatic.  LUNGS:  Demonstrate decreased breath sounds globally with small rhonchi in the inferior right lung, which is clear somewhat with cough.  HEART:  Regular rate and rhythm.  ABDOMEN:  Soft and nontender.  The liver and spleen are not palpably enlarged.  There is a large midline defect covered by skin, of which the fascia has numerous holes poking through.  These small hernias are of approximately 2 cm in diameter at the largest and are nontender.  GU:  The penis is circumcised and there are no lesions, plaques, masses or deformities.  There is some tenderness to palpation near the meatus where 20-French Foley catheter is in place.  Testes are bilaterally descended and there are no masses or tenderness.  There is bilateral mild atrophy.  Epididymidis are grossly within normal limits bilaterally.  Spermatic cords are grossly within normal limits.  There are no palpable inguinal hernias.  RECTAL:  The prostate is mildly enlarged with a small focal firm area in the midline near the apex.  There is however no other focal nodules.  The prostate is grossly approximately 35 to 40 g and is globally firm.  Rectal sphincter tone is grossly within normal limits and there is stool in the rectal vault.  EXTREMITIES:  Demonstrate no cyanosis, clubbing or edema.  There is dark red urine in the Foley bag collection.,LABORATORY EXAM:,  Review of laboratory from outside facility demonstrates creatinine of 2.38 with BUN of 42.  Additionally, laboratory exam demonstrates a grossly normal serum cortisol, ACTH, potassium, aldosterone level during lesion workup.  CT scan was reviewed from outside facility, report states there is left kidney atrophy without hydro or stones and there is thickened left bladder wall and posterior margins with a balloon inflated in the prostate at the time of the exam.  There is a 3.1 cm right heterogeneous adrenal nodule and there are no upper tract lesions or stones noted.,IMPRESSION:,  Bladder cancer.,PLAN:  ,The patient will undergo a completion TURBT on 03/20/08 with bilateral retrograde pyelograms at the time of surgery.  Preoperative workup and laboratory as well as paper work were performed in clinic today with Dr. Y. The patient will be scheduled for anesthesia preop.  The patient will have urine culture redrawn from his Foley or penis at the time of preoperative evaluation with anesthesia.  The patient was counseled extensively approximately 45 minutes on the nature of his disease and basic prognostic indicators and need for additional workup and staging.  The patient understands these instructions and also agrees to quit smoking prior to his next visit.  This patient was seen in evaluation with Dr. Y who agrees with the impression and plan.","urology, retrograde pyelogram, bladder biopsy, muscularis propria, bladder cancer, gross hematuria, bladder wall, ct scan, bladder, hematuria,
         """,
        """Patient with hypertension, syncope, and spinal stenosis - for recheck.", SOAP / Chart / Progress Notes, Hypertension - Progress Note ,"SUBJECTIVE:,  The patient is a 78-year-old female who returns for recheck.  She has hypertension.  She denies difficulty with chest pain, palpations, orthopnea, nocturnal dyspnea, or edema.,PAST MEDICAL HISTORY / SURGERY / HOSPITALIZATIONS:,  Reviewed and unchanged from the dictation on 12/03/2003.,MEDICATIONS:  ,Atenolol 50 mg daily, Premarin 0.625 mg daily, calcium with vitamin D two to three pills daily, multivitamin daily, aspirin as needed, and TriViFlor 25 mg two pills daily.  She also has Elocon cream 0.1% and Synalar cream 0.01% that she uses as needed for rash.,ALLERGIES:  ,Benadryl, phenobarbitone, morphine, Lasix, and latex.,FAMILY HISTORY / PERSONAL HISTORY: , Reviewed.  Mother died from congestive heart failure.  Father died from myocardial infarction at the age of 56.  Family history is positive for ischemic cardiac disease.  Brother died from lymphoma.  She has one brother living who has had angioplasties x 2.  She has one brother with asthma.,PERSONAL HISTORY:,  Negative for use of alcohol or tobacco.,REVIEW OF SYSTEMS:,Bones and Joints:  She has had continued difficulty with lower back pain particularly with standing which usually radiates down her right leg.  She had been followed by Dr. Mills, but decided to see Dr. XYZ who referred to her Dr Isaac.  She underwent several tests.  She did have magnetic resonance angiography of the lower extremities and the aorta which were normal.  She had nerve conduction study that showed several peripheral polyneuropathy.  She reports that she has myelogram last week but has not got results of this.  She reports that the rest of her tests have been normal, but it seems that vertebrae shift when she stands and then pinches the nerve.  She is now seeing Dr. XYZ who comes to Hutchison from KU Medical Center, and she thinks that she probably will have surgery in the near future.,Genitourinary:  She has occasional nocturia.,PHYSICAL EXAMINATION:,Vital Signs:  Weight:  227.2 pounds.  Blood pressure:  144/72.  Pulse:  80.  Temperature:  97.5 degrees.,General Appearance:  She is an elderly female patient who is not in acute distress.,Mouth:  Posterior pharynx is clear.,Neck:  Without adenopathy or thyromegaly.,Chest:  Lungs are resonant to percussion.  Auscultation reveals normal breath sounds.,Heart:  Normal S1 and S2 without gallops or rubs.,Abdomen:  Without masses or tenderness to palpation.,Extremities:  Without edema.,IMPRESSION/PLAN:,1.  Hypertension.  She is advised to continue with the same medication.,2.  Syncope.  She previously had an episode of syncope around Thanksgiving.  She has not had a recurrence of this and her prior cardiac studies did not show arrhythmias.,3.  Spinal stenosis.  She still is being evaluated for this and possibly will have surgery in the near future.","soap / chart / progress notes, progress note, hypertension, spinal stenosis, syncope, spinal, stenosis, infarction, orthopnea,
         """,
        """" Patient with hip pain, osteoarthritis, lumbar spondylosis, chronic sacroiliitis, etc.", SOAP / Chart / Progress Notes, Chiropractic Progress Note ,"CHIEF COMPLAINT:  ,Hip pain.,HISTORY OF PRESENTING ILLNESS:  ,The patient is a very pleasant 41-year-old white female that is known to me previously from our work at the Pain Management Clinic, as well as from my residency training program, San Francisco. We have worked collaboratively for many years at the Pain Management Clinic and with her departure there, she has asked to establish with me for clinic pain management at my office.  She reports moderate to severe pain related to a complicated past medical history.  In essence, she was seen at a very young age at the clinic for bilateral knee and hip pain and diagnosed with bursitis at age 23.  She was given nonsteroidals at that time, which did help with this discomfort.  With time, however, this became inadequate and she was seen later in San Francisco in her mid 30s by Dr. V, an orthopedist who diagnosed retroverted hips at Hospital.  She was referred for rehabilitation and strengthening.  Most of this was focused on her SI joints.  At that time, although she had complained of foot discomfort, she was not treated for it.  This was in 1993 after which she and her new husband moved to the Boston area, where she lived from 1995-1996.  She was seen at the Pain Center by Dr. R with similar complaints of hip and knee pain.  She was seen by rheumatologists there and diagnosed with osteoarthritis as well as osteophytosis of the back.  Medications at that time were salicylate and Ultram.,When she returned to Portland in 1996, she was then working for Dr. B.  She was referred to a podiatrist by her local doctor who found several fractured sesamoid bones in her both feet, but this was later found not to be the case.  Subsequently, nuclear bone scans revealed osteoarthritis.  Orthotics were provided.  She was given Paxil and Tramadol and subsequently developed an unfortunate side effect of grand mal seizure.  During this workup of her seizure, imaging studies revealed a pericardial fluid-filled cyst adhered to her ventricle.  She has been advised not to undergo any corrective or reparative surgery as well as to limit her activities since.  She currently does not have an established cardiologist having just changed insurance plans.  She is establishing care with Dr. S, of Rheumatology for her ongoing care.  Up until today, her pain medications were being written by Dr. Y prior to establishing with Dr. L.,Pain management in town had been first provided by the office of Dr. F. Under his care, followup MRIs were done which showed ongoing degenerative disc disease, joint disease, and facet arthropathy in addition to previously described sacroiliitis.  A number of medications were attempted there, including fentanyl patches with Flonase from 25 mcg titrated upwards to 50 mcg, but this caused oversedation.  She then transferred her care to Ab Cd, FNP under the direction of Dr. K.  Her care there was satisfactory, but because of her work schedule, the patient found this burdensome as well as the guidelines set forth in terms of monthly meetings and routine urine screens.  Because of a previous commitment, she was unable to make one unscheduled request to their office in order to produce a random urine screen and was therefore discharged.,PAST MEDICAL HISTORY:  ,1.  Attention deficit disorder.,2.  TMJ arthropathy.,3.  Migraines.,4.  Osteoarthritis as described above.,PAST SURGICAL HISTORY:,1.  Cystectomies.,2.  Sinuses.,3.  Left ganglia of the head and subdermally in various locations.,4.  TMJ and bruxism.,FAMILY HISTORY:  ,The patient's father also suffered from bilateral hip osteoarthritis.,MEDICATIONS:,1.  Methadone 2.5 mg p.o. t.i.d.,2.  Norco 10/325 mg p.o. q.i.d.,3.  Tenormin 50 mg q.a.m.,4.  Skelaxin 800 mg b.i.d. to t.i.d. p.r.n.,5.  Wellbutrin SR 100 mg q.d.,6.  Naprosyn 500 mg one to two pills q.d. p.r.n.,ALLERGIES: , IV morphine causes hives.  Sulfa caused blisters and rash.,PHYSICAL EXAMINATION: , A well-developed, well-nourished white female in no acute distress, sitting comfortably and answering questions appropriately, making good eye contact, and no evidence of pain behavior.,VITAL SIGNS:  Blood pressure 110/72 with a pulse of 68.,HEENT:  Normocephalic.  Atraumatic.  Pupils are equal and reactive to light and accommodation.  Extraocular motions are intact.  No scleral icterus.  No nystagmus.  Tongue is midline.  Mucous membranes are moist without exudate.,NECK:  Free range of motion without thyromegaly.,CHEST:  Clear to auscultation without wheeze or rhonchi.,HEART:  Regular rate and rhythm without murmur, gallop, or rub.,ABDOMEN:  Soft, nontender.,MUSCULOSKELETAL:  There is musculoskeletal soreness and tenderness found at the ankles, feet, as well as the low back, particularly above the SI joints bilaterally.  Passive hip motion also elicits bilateral hip pain referred to the ipsilateral side.  Toe-heel walking is performed without difficulty.  Straight leg raises are negative.  Romberg's are negative.,NEUROLOGIC:  Grossly intact.  Intact reflexes in all extremities tested.  Romberg is negative and downgoing.,ASSESSMENT:,1. Osteoarthritis.,2. Chronic sacroiliitis.,3. Lumbar spondylosis.,4. Migraine.,5. TMJ arthropathy secondary to bruxism.,6. Mood disorder secondary to chronic pain.,7. Attention deficit disorder, currently untreated and self diagnosed.,RECOMMENDATIONS:,1. Agree with Rheumatology referral and review.  I would particularly be interested in the patient pursuing a bone density scan as well as thyroid and parathyroid studies.,2. Given the patient's previous sulfa allergies, we would recommend decreasing her Naprosyn usage."
         """,
        """Acne with folliculitis., SOAP / Chart / Progress Notes, Acne - SOAP ,"SUBJECTIVE:,  The patient is a 49-year-old white female, established patient to Dermatology, last seen in the office on 08/10/2004.  She comes in today for reevaluation of her acne plus she has had what she calls a rash for the past two months now on her chest, stomach, neck, and back.  On examination, this is a flaring of her acne with small folliculitis lesions.  The patient has been taking amoxicillin 500 mg b.i.d. and using Tazorac cream 0.1, and her face is doing well, but she has been out of her medicine now for three days also.  She has also been getting photofacials at Healing Waters and was wondering about what we could offer as far as cosmetic procedures and skin care products, etc.  The patient is married.  She is a secretary.,FAMILY, SOCIAL, AND ALLERGY HISTORY:,  She has hay fever, eczema, sinus, and hives.  She has no melanoma or skin cancers or psoriasis.  Her mother had oral cancer.  The patient is a nonsmoker.  No blood tests.  Had some sunburn in the past.  She is on benzoyl peroxide and Daypro.,CURRENT MEDICATIONS:,  Lexapro, Effexor, Ditropan, aspirin, vitamins.,PHYSICAL EXAMINATION:,  The patient is well developed, appears stated age.  Overall health is good.  She has a couple of acne lesions, one on her face and neck but there are a lot of small folliculitis-like lesions on her abdomen, chest, and back.,IMPRESSION:,  Acne with folliculitis.,TREATMENT:,1.  Discussed condition and treatment with the patient.,2.  Continue the amoxicillin 500 mg two at bedtime.,3.  Add Septra DS every morning with extra water.,4.  Continue the Tazorac cream 0.1; it is okay to use on back and chest also.,5.  Referred to ABC clinic for an aesthetic consult.  Return in two months for followup evaluation of her acne.","soap / chart / progress notes, acne with folliculitis, tazorac cream, acne, dermatology, tazorac, cream, folliculitis,
         """,
       # """Patient presents for treatment of suspected rheumatoid arthritis., Rheumatology, Rheumatoid Arthritis - H&P ,"CHIEF COMPLAINT:,  This 26 year old male presents today for treatment of suspected rheumatoid arthritis.  Associated signs and symptoms include aching, joint pain, and symmetrical joint swelling bilateral.  Patient denies any previous history, related trauma or previous treatments for this condition.  Condition has existed for 2 weeks.  He indicates the problem location is the right hand and left hand.  Patient indicates no modifying factors.  Severity of condition is slowly worsening.  Onset was unknown.,ALLERGIES:,  Patient admits allergies to aspirin resulting in GI upset, disorientation.,MEDICATION HISTORY: , Patient is currently taking amoxicillin-clavulanate 125 mg-31.25 mg tablet, chewable medication was prescribed by A. General Practitioner MD, Adrenocot 0.5 mg tablet medication was prescribed by A. General Practitioner MD.,PAST MEDICAL HISTORY:,  Past medical history is unremarkable.,PAST SURGICAL HISTORY: , Patient admits past surgical history of (+) appendectomy in 1989.,FAMILY HISTORY: , Patient admits a family history of rheumatoid arthritis associated with maternal grandmother.,SOCIAL HISTORY:  ,Patient denies alcohol use.  Patient denies illegal drug use.  Patient denies STD history.  Patient denies tobacco use.,REVIEW OF SYSTEMS: , Neurological: (+) paralysis Musculoskeletal: (+) joint pain (+) joint swelling (+) stiffness Cardiovascular: (+) ankle swelling Neurological: (-) numbness,Musculoskeletal: (-) back pain (chronic) (-) decreased ROM (-) episodic weakness,Cardiovascular: (-) chest pressure Respiratory: (-) breathing difficulties, respiratory symptoms (-) sleep apnea,PHYSICAL EXAM: , BP Standing:  120/84 HR:  79 Temp:  98.6 Height:  5 ft.  8 in.  Weight:  168 lbs.  Patient is a 26 year old male who appears pleasant, in no apparent distress, his given age, well developed, well nourished and with good attention to hygiene and body habitus.  Skin:  No skin rash, subcutaneous nodules, lesions or ulcers observed.  Palpation of skin shows no abnormalities.,HEENT:  Inspection of head and face shows no abnormalities.  Hair growth and distribution is normal.  Examination of scalp shows no abnormalities.  Conjunctiva and lids reveal no signs or symptoms of infection.  Pupil exam reveals round and reactive pupils without afferent pupillary defect.  Ocular motility exam reveals gross orthotropia with full ductions and versions bilateral.  Bilateral retinas reveal normal color, contour, and cupping.  Inspection of ears reveals no abnormalities.  Otoscopic examination reveals no abnormalities.  Examination of oropharynx reveals no abnormalities and tissues pink and moist.  ENT:  Inspection of ears reveals no abnormalities.  Examination of larynx reveals no abnormalities.  Inspection of nose reveals no abnormalities.,Neck:  Neck exam reveals neck supple and trachea that is midline, without adenopathy or crepitance palpable.  Thyroid examination reveals no abnormalities and smooth and symmetric gland with no enlargement, tenderness or masses noted.  Lymphatic:  Neck lymph nodes are normal.,Respiratory:  Assessment of respiratory effort reveals even respirations without use of accessory muscles and no intercostal retractions noted.  Chest inspection reveals chest configuration non-hyperinflated and symmetric expansion.  Auscultation of lungs reveals clear lung fields and no rubs noted.,Cardiovascular:  Heart auscultation reveals normal S1 and S2 and no murmurs, gallop, rubs or clicks.  Examination of peripheral vascular system reveals full to palpation, varicosities absent, extremities warm to touch and no edema.,Abdomen:  Abdominal contour is slightly rounded.  Abdomen soft, nontender, bowel sounds present x 4 without palpable masses.  Palpation of liver reveals no abnormalities.  Palpation of spleen reveals no abnormalities.,Musculoskeletal:  Gait and station examination reveals normal arm swing, with normal heel-toe and tandem walking.  Inspection and palpation of bones, joints and muscles is unremarkable.  Muscle strength is 5/5 for all groups tested.  Muscle tone is normal.,Neurologic/Psychiatric:  Psychiatric:  Oriented to person, place and time.  Mood and affect normal and appropriate to situation.  Testing of cranial nerves reveals no deficits.  Coordination is good.  Touch, pin, vibratory and proprioception sensations are normal.  Deep tendon reflexes normal.,TEST & X-RAY RESULTS:,  Rheumatoid factor:  52 U/ml.  Sed rate:  31 mm/hr.  C4 complement:  19 mg/dl.,IMPRESSION: , Rheumatoid arthritis.,PLAN:,  ESR ordered; automated.  Ordered RBC.  Ordered quantitative rheumatoid factor.  Return to clinic in 2 week (s).,PRESCRIPTIONS:,  Vioxx Dosage:  12.5 mg tablet Sig:  BID Dispense:  30 Refills:  2 Allow Generic:  No",
        # """,
       # """Comprehensive Clinical Psychological Evaluation as part of a Disability Determination action., Psychiatry / Psychology, Psychological Evaluation ,"COMPREHENSIVE CLINICAL PSYCHOLOGICAL EVALUATION,CURRENT MEDICATIONS:,  Nexium 4 mg 4 times per day, Propanolol 10 mg 4 times a day, Spironolactone 100 mg 3 times per day, Lactulose 60 cc's 3 times a day.,GENERAL OBSERVATIONS:  ,Mr. Abc, a 54-year-old black married male who was referred for a Comprehensive Clinical Psychological Evaluation as part of a Disability Determination action.  Mr. Abc arrived five minutes late for his scheduled appointment.  He was accompanied to the office by his sister-in-law who drove him to the appt.  Mr. Abc currently does not receive Disability benefits.  This is the first time he has filed for Disability.  The Authorization form listed Mr. Abc's current complaints as ""cirrhosis of the liver and mental issues.""  Mr. Abc was well groomed and wore casual attire.  He looked older than his stated age.  The whites of his eyes were very jaundiced.  His posture was slightly stooped and his gait was slow.  He was winded after walking up the stairs.  Psychomotor activity was retarded.  Mr. Abc was cooperative throughout the interview.  Although he appeared to be answering most questions to the best of his ability, he appeared to be minimizing his emotional distress.  ,PRESENT ILLNESS: , Most information was provided by Mr. Abc who appeared to be a fairly reliable source.  His information was supplemented by review of his medical records.  Mr. Abc has applied for Federal Disability benefits believing that he qualifies based on his cirrhosis of the liver and his cognitive dysfunction.  Mr. Abc was diagnosed with cirrhosis in 1991.  His condition has worsened to the point that he is experiencing liver failure and is awaiting a liver transplant.  He stated that his main symptom is extreme fatigue.  He has no energy and is unable to engage in many activities.  Over the past year he was admitted to the hospital four times for confusion and bizarre behavior.  He stated that his sister-in-law and his wife told him that he had become violent and he fought with the Sherriff who was trying to take him to the hospital.  He has no memory of this.  Mr. Abc stated that he was hospitalized one time.  Actually he had begun having problems with confusion in July of 2004 and he has been treated four times since that time.  According to his medical records, he was found wandering outside of his home.  He was apparently delusional believing that a tree branch was a doorknob.  Mr. Abc also suffers from edema and swelling in his legs and his feet.  Mr. Abc attempted to return to work and found that he was unable to do his job due to the necessity of walking one-quarter mile from the front to the back of the plant.  He was unable to walk very far without becoming fatigued.  He had instances where he had passed out after becoming faint.  He had trouble at work sitting for very long because his feet swelled.  He was unable to lift the required 10 pounds of medication boxes.  When he found himself unable to do his regular job, he tried another job at the same plant but was unable to do that job.  He also became confused easily at work.  His doctor advised him to quit and then he did so in March of this year.  In addition to his cognitive symptoms, Mr. Abc has had some disturbance in mood as well.  He related that he feels very sad since he lost his job.  A lot of his self-esteem came from working.  He worries about financial problems.  His sleep has been disturbed.  He sleeps four to five hours a night with trouble falling asleep and frequent awakening in the middle of the night.  His appetite is fair.  ,PERSONAL, FAMILY AND SOCIAL HISTORY:,  Mr. Abc completed the 11th grade. He went on to get his GED in 1971.  He stated that he has never failed a grade and he has no history of a learning disability.  He received no special education services.  His grades were Bs and Cs. He stated that he was suspended from school one time for fighting but got along well in general. Mr. Abc is currently unemployed.  His last job was at Baxter Health Care where he worked for four years.  It was his longest place of employment.  He quit in March of 2005 because of fatigue and inability to perform the necessary job duties.  He denies that he was ever fired from a job and he reported good work relationships.  Mr. Abc has been married for two years. He has no prior marriages.  He has one daughter age 13.  He currently lives with his wife. Has been at his current address for four years.  ,HISTORY OF OTHER PERTINENT MEDICAL EVENTS: , Mr. Abc has cirrhosis of the liver, hepatitis C, hepatic encephalopathy, and gastroesophageal reflux disease, and hypertension. Surgeries include a cardiac catheterization in 2001, a liver biopsy in 2003.  Over the past year he has been hospitalized four times due to confusion and bizarre behaviors stemming from his liver failure.  ,DAILY ACTIVITIES AND FUNCTIONING:  ,Mr. Abc stated that he tries to do things but he has been severely restricted due to his extreme fatigue.  He enjoys reading and does it regularly.  He tries to help his wife with the household chores as he can.  He has washed dishes, cooked, mopped, dusted, vacuumed and has done laundry occasionally over the past month but not as much as he used to.  He stated that he used to mow the yard and do yard work but he can no longer do it because of his extreme fatigue.  He has given up driving all together and he no longer goes out alone.  He spends most days at home.  He enjoys going to church and he prays daily.  ,MENTAL HEALTH HISTORY: , Mr. Abc has never been diagnosed or treated for a mental health disorder.  He denied any history of mental health problems in his family.  He stated that he was evaluated one time earlier this year by a psychiatrist to determine his suitability for a liver transplant.  He was approved and he is now on the waiting list to receive a liver.  ,SUBSTANCE USE HISTORY:  ,Mr. Abc has a history of substance use beginning in his teenage years.  He has used alcohol, marijuana and cocaine.  He stated that he only used the marijuana and cocaine a few times when he was young but he continued using alcohol until recently.  His alcohol use became problematic and he was arrested for DWI three times.  He attended AA and the DART program.  Mr. Abc stated that he has been clean for eight years and five months.
       #  """,
    # """ Psychiatric evaluation for ADHD, combined type.", Psychiatry / Psychology, Psychiatric Evaluation - 3 ,"IDENTIFICATION OF PATIENT: , ABCD is an 8-year-old Hispanic male currently in the second grade.,CHIEF COMPLAINT/HISTORY OF PRESENT ILLNESS: , ABCD presents to this visit with his mother, Xyz, and her significant other, Pqr.  Circumstances leading to this admission:  In the past, ABCD has been diagnosed and treated for ADHD, combined type, and has been on Concerta 54 mg one p.o. q.8h.  Since he has been on the 54 mg, mother has concerns because he has not been sleeping well at night, consistently he is staying up until 12:00 or 1:00, and he is not eating the noonday meal and not that much for supper.  ABCD is also complaining of headaches when he takes the medication.  Mother reports that on the weekends he is off the medication.  She does notice that his sisters become more irritated with him and say he is either hitting them or bothering them and he will say, ""It's an accident.""  She sees him as impulsive on the weekends, but is not sure if this just isn't ""all boy."",Mother reports ABCD has been on medication since kindergarten.  Currently, the teachers say he is able to pay attention and he is well behaved in school.  Prior to being on medication, there were issues with the teachers saying he was distractible and had difficulty paying attention.,He had a psychological evaluation done on 07/16/06 by Dr. X, in which he was diagnosed with ADHD, combined type; ODD; rule out depressive disorder, NOS; rule out adjustment disorder with depressed mood; and rule out adjustment disorder with mixed features of conduct.  He also has seen XYZ, LCSW, in the past for outpatient therapy.,ABCD's mother, A, as well as her significant other, R, and his teachers are not convinced that he needs his medication and would like to either trial him off or trial him on a lower dose.,REVIEW OF SYSTEMS:,Sleep:  As stated before, he is having much difficulty on a consistent basis falling asleep.  It is 12:00 to 1:00 a.m. before he falls to sleep.  When he was on the 36 mg of Concerta, he was able to fall asleep without difficulty.  On the weekends, he is also having difficulty falling asleep, even though he is not taking the medication.,Appetite:  He will eat breakfast and supper, but not much lunch, if any at all.  He has not lost weight that mother is aware of, nor is he getting more sick than normal.,Mood control:  Mother reports he has not been aggressive since he has been on the medication, nor is he getting in trouble at school for aggression or misbehavior.  The only exception to this is he gets in occasional fights with his sisters.  ABCD denies visual or auditory hallucinations or racing thoughts.  He reports his thoughts are sometimes bad because he says sometimes he thinks of the ""S"" word.,Energy:  Mother reports a lot of energy.,Pain:  ABCD denies any pain in his body.,Suicidal or homicidal thoughts:  He denies suicidal thoughts or plan to hurt himself or anyone else.,PAST TREATMENT AND/OR MEDICATIONS:,ABCD was originally tried on Ritalin in kindergarten, and he has been on Concerta since 07/14/06.  He has received outpatient therapy from XYZ, LCSW.  He is currently not in outpatient therapy.,FAMILY PSYCHIATRIC HISTORY:,Mother reports that on her side of the family she is currently being assessed for mood disorder/bipolar.  She reports she has significant moodiness episodes and believes in the past she has had a manic episode.  She is currently not on medication.  She does not know of anyone else in her family, with the exception of she said her father's behavior was ""weird.""  Biological father's side of the family, mother reports father was very impulsive.  He had anger issues.  He had drug and alcohol issues.  He was in jail for three years for risky behavior.  There was also domestic violence when mother was married to father.,FAMILY AND SOCIAL HISTORY:,Biological mother and father were married for five years.  They divorced when ABCD was 2-1/2 years of age.  Currently, father has been deported back to Mexico.  He last saw ABCD in March 2006 for one day when they went down to AAAA.  He does call on special holidays and his birthday.  Contact is brief, but so far has been consistent.  Mother is currently seeing R, a significant other, and has been seeing him for the last seven months.  ABCD had a good relationship with R.  ABCD has an older sister, M, age 9, who they describe as very gifted and creative without attention issues or oppositional issues, and a younger sister, S, age 7, who mother describes as ""all wisdom."",PREGNANCY:,  Mother reports her pregnancy was within normal limits as well as the labor and birth; although, she was exposed to domestic violence while ABCD was in utero.  She did not use drugs or alcohol while she was pregnant.,DEVELOPMENTAL MILESTONES:,  Developmental milestones were all met on time, although ABCD has had speech therapy since he was young.,PHYSICAL ABUSE:,  Mother and ABCD deny any history of physical or sexual abuse or emotional abuse, with the exception of exposure to domestic violence when he was very young, age 2 and before.,DISCIPLINE PROBLEMS:,  Mother reports ABCD was a very cuddly infant and could sleep well.  As a toddler, he was all over the place, climbing and always busy.  Elementary school:  In kindergarten, the teacher said it was very emphatic that he needed medication because he could not focus or sit still or listen.  ABCD has no history of fire setting or abuse to animals.  He does not lie more than other kids his age and he does not have any issues with stealing.,PAST DRUG AND ALCOHOL HISTORY:,  Noncontributory.,MEDICAL STATUS AND HISTORY:,  ABCD has no known drug allergies.  He has no history of heart murmur, heart defect of other heart problems.  No history of asthma, seizures or head injuries.  He no medical diagnosis and he has ever spent an overnight in a medical hospital.,SCHOOL:,  When I asked ABCD whether he likes school, he stated, ""No.""  His grades are okay, per mother.  He does have an IEP for the ADHD, but she does not believe he has a learning disability.  Behavior problems:  He currently is not having any behavior problems in the school.  He reports he does not get along with his teachers because they tell him what to do.  Strengths:  He reports he loves to read and he can focus and concentrate on his reading and he dislikes centers.,RELATIONSHIPS:,  He reports he has best friends.  He named two, D and B, and he does have a friend that is a girl named Kim.  When asked if church or God were important to him, he stated, ""God is.""  He is in a Roman Catholic family and that is an important aspect of his life.,WORK HISTORY:,  In the home, he has chores of taking out the trash.,LEGAL:,  He has not been involved in the legal system.,SUPPORT SYSTEMS:,  When asked if he feels safe in his home, he stated, ""Yes.""  When asked who he talks to if he is hurt or upset, he stated, ""Mom.""  (At first, he said video games, but then he said mom).,TALENTS AND GIFTS:,  He is good at basketball, video games, and reading books.,MENTAL STATUS EXAM:,  This was a very long appointment, approximately two hours in length, due to mother and significant other had many questions.  ABCD kept himself occupied throughout and was very well behaved throughout the session.  He had some significant memory responses in that he remembered the last holiday was Martin Luther King Day, which is somewhat unusual for a child his age, but he could only recall one of three items after five minutes.  Distractibility and attention:  He, at times, was very mildly distracted, but otherwise did not appear hyperactive.  His judgment was adequate.  When asked what he would do if there was a fire in his house, he said, ""Get out!""  Insight was poor to adequate.  Fund of information was good.  When asked who the president was, he said, ""George Washington."" Intelligence is probably average to above average.  Speech was normal.  He had some difficulty with abstract thinking.  He could not see any similarities between an orange and an apple, but was able to see similarities of wheels between an airplane and a bicycle.  On serial 7's he could do 100 minus 7, but then unable to subtract any of the others, but he completed serial 3's very rapidly.  When given three commands in a row, he used his left hand instead of his right hand, but followed the last two commands correctly.  Appearance was casual.  Hygiene was good.  Attitude was cooperative.  Speech was normal.  Psychomotor was between normal and slightly hyperactive.  Orientation was x2.  Attention/concentration was intact.  Memory was intact at times and then had some memory recall problems with three words.  Mood was euthymic.  Affect was bright.  He has no suicidal or homicidal/violence risks.  Perceptions were normal.  Thought process logical.  Thought content normal.  Disassociation none.  Sleep:  He is having some insomnia.  Appetite/eating are decreased.,STRENGTHS AND SUPPORTS:,  He has a strong support system in his mother, grandmother, and mother's significant other, Richard.  He has good health.  He has shown gain from past treatment.  He has a sense of humor and a positive relationship with his mother and her significant other, as well as good school behavior.",

   # """,
    # """Psychiatric Assessment of a patient with bipolar and anxiety disorder having posttraumatic stress syndrome., Psychiatry / Psychology, Psychiatric Assessment ,"IDENTIFICATION OF PATIENT: , This is a 31-year-old female who was referred by herself.  She was formerly seen at Counseling Center.  She is a reliable historian.,CHIEF COMPLAINT:,  ""I'm bipolar and I have severe anxiety disorder.  I have posttraumatic stress syndrome."" ,HISTORY OF PRESENT ILLNESS: , At age 19, Ms. Abc had a recurrence of memories.  Her father had molested her, and the memories returned.  In 1992, at the age of 18, she entered her first abusive marriage.  She was beaten and her husband shared her sexually with his friends.  This lasted until age 24.  The second marriage was age 26, her second husband was a drug abuser and ""he slapped me around.""  She had two children during that marriage.  In 2001, she was married in Indiana to a military man.  This was her third marriage and she stated, ""This marriage is good.""  She had EMDR in Indiana when she was being treated for Posttraumatic Stress Disorder.  ,Historically, her first husband threw her down the stairs at age 21, and she had a miscarriage.  Her sexual abuse began at age 5, and at that time she lost interest in other activities that normal school children have.  Currently, she is unable to have sex with the lights on.  She states, ""Sometimes I hurt all over.""  Her husband was deployed three days ago, on April 21, to a foreign theater of operations.  She has panic attacks every day.,Review of symptoms shows her to have physiological distress at the memory of her trauma, she has psychological distress, and this comes about when she smells Old Spice aftershave.  She does not avoid thoughts of her trauma, but she avoids the perpetrators and placements.  She is not unable to recall details of her trauma.  She does feel detached and isolated.  She has restrictive range of affect and she had a foreshortened future.  She also had a loss of interest in things, starting at age 5.  She has anger, which is uncontrollable at times, she has poor sleep, she has nightmares, flashbacks, she is hypervigilant, she has exaggerated startle reflex, and with respect to concentration, she says, ""I don't do as good as I can.""  Further review of symptoms shows her to have periods of constant cleaning and increased sex drive.  She also has had euphoria, poor judgment, distractibility, and inability to concentrate.  She has been irritable.  She has had a decreased need for sleep, which lasts for six or seven days.  She had racing thoughts, rapid speech, but has not had grandiosity.  These symptoms of mania occurred in the last week of November 2005 and lasted for seven days from, which she was not hospitalized.  Furthermore, she endorses the following symptoms:  She states, ""When I'm depressed, I have neck pain, jaw pain, abdominal pain.  I have migraines and urinary tract pain.""  She also complains of chest pain, pain during sex, and excess pain during her menstrual period.  She has an increased gag reflex, which has caused her to have emesis.  She states it is easy to choke.  She has had physical symptoms, ""for as long as I can remember,"" and she states, ""I've felt like crap most of my life,""  ""it affects my marriage.""  She has also admitted to having nausea and vomiting, with excess gas.  She has constipation and she cannot eat certain foods, mainly broccoli and cauliflower, and she does not have diarrhea.  She states that sex is only important to her in mania.  Otherwise, she has no desire.  She has had irregular periods for two or three weeks at a time.  She has had no episodes of excess bleeding.  She has had no paralysis, no balance issues, no diplopia, no seizures, no blindness, no deafness, no amnesia, no loss of consciousness, but she does have a lump in her throat on occasion.  Currently, she is sleeping from 10 p.m. to 3 a.m., and that is under the influence of Lunesta.  Her energy is ""not good.  Her appetite is ""I'm craving crap,"" stating that she wants to eat carbohydrates.  Concentration is poor today.  She feels worthless, hopeless, and guilty.  Her self-esteem is ""I don't have any.""  She has no anhedonia, and she has no libido.  She also has had feelings of chronic emptiness.  She feels abandoned.  She has had unstable relationships.  She self-mutilated, but she stopped at age 22.  She has trouble controlling her anger.  She did not have stress-related paranoia or dissociative phenomena, but she did have those during the sexual transgressions when she was a child.  She has no identity disturbance.  ,CURRENT MEDICATIONS: , Seroquel 700 mg p.o. q.d.; Wellbutrin XL 300 mg p.o. q.d.; Desyrel 100 mg p.o. q.h.s.; Ativan p.r.n. dosage unknown.  In the past, she has been on Prozac, Paxil, lithium, Depakote, Depakene, and Zoloft. ,PSYCHIATRIC HISTORY: , She saw Dr. B.  She saw Chris.  She is diagnosed with Posttraumatic Stress Disorder, depression, and Bipolar Disorder.  She had counseling in Indiana in 2001.  She had inpatient treatment in Indiana in 2001 also, at age 19.  She had three suicide attempts.  At age 14, she took too many aspirin; the second one was at age 19, she took pain medication and sleep medication; and when she discussed her third suicide attempt, she began to cry and would not speak of it any more.  She has had no psychological testing.  ,MEDICAL HISTORY: , Significant for migraines, hyperactive and gag reflex.  She states she has had cardiovascular workups due to panic disorder, but nothing was found.  She also has astigmatism.  She states she has stomach pain and may have irritable bowel syndrome, and she had had recurrent kidney infections with a stent in the right kidney during one of her pregnancy.  She has no history of head injury or MRI test of the brain.  No history of EEG, seizures, thyroid problems, or asthma.  There are no drug allergies.  She has never had an EKG.  She does have musculoskeletal problems and has arthritis-like joint pains on occasion.  She has had ear infections and sinus infections intermittently.  Hearing test was normal.  She is currently not pregnant.  She saw her gynecologist four months ago at Elmendorf Air Force Base.  ,Surgical history is significant for having a tubal ligation at age 27, an appendectomy at age 19.  She had surgery on her right ovary due to pain, a cyst was found; the date on that is unknown.  ,She has no hypertension, no diabetes, no glaucoma.,FAMILY HISTORY: , Significant for her paternal grandmother not being mentally competent.  Her mother was depressed and was treated.  Her mother is currently age 55.  She has a paternal grandmother who may have had Schizophrenia.  There is also a family history of the paternal grandfather using substance.  He was ""an extreme alcoholic.""  She had maternal aunts who used alcohol, and a maternal uncle use alcohol to excess.  The maternal uncle committed suicide; he drowned himself.  ,There is no family history of bipolar disorder, anxiety, nor attention deficit, mental netardation, Tourette's syndrome, or learning disabilities. ,Medical history in the family is significant for her son, age 4, who is having seizures ruled out.  Her mother and two maternal aunts have thyroid disease.  She has a brother, age 32, with diabetes, a maternal uncle with heart disease, and several paternal great aunts had breast cancer.  There is no family history of hypertension.,ABUSE HISTORY: , Significant for being physically abused by her father, her first husband, and her second husband.  She was sexually abused by her father from age 5 to age 18.  She states, ""my first husband gave me away for four years to his friends to be used sexually.""  She was emotionally abused by her mother, father, and both of her first two husbands.  She was neglected by her mother and her father.  She never witnessed domestic violence.  She has not witnessed traumatic events.  ,SUBSTANCE ABUSE: , Significant for having used nerve pills, but she stated she has not used them excessively
   #  """,
  #  """ A 45-year-old white male with a history of schizophrenia and AIDS.  He was admitted for disorganized and assaultive behaviors while off all medications for the last six months., Psychiatry / Psychology, Psych Consult - Schizophrenia ,"IDENTIFYING DATA:,  The patient is a 45-year-old white male.  He is unemployed, presumably on disability and lives with his partner.,CHIEF COMPLAINT: , ""I'm in jail because I was wrongly arrested."" The patient is admitted on a 72-hour Involuntary Treatment Act for grave disability.,HISTORY OF PRESENT ILLNESS: , The patient has minimal insight into the circumstances that resulted in this admission.  He reports being diagnosed with AIDS and schizophrenia for some time, but states he believes that he has maintained his stable baseline for many months of treatment for either condition.  Prior to admission, the patient was brought to Emergency Room after he attempted to shoplift from a local department store, during which he apparently slapped his partner.  The patient was disorganized with police and emergency room staff, and he was ultimately detained on a 72-hour Involuntary Treatment Act for grave disability.,On the interview, the patient is still disorganized and confused.  He believes that he has been arrested and is in jail.  Reports a history of mental health treatment, but denies benefiting from this in the past and does not think that it is currently necessary.,I was able to contact his partner by telephone.  His partner reports the patient is paranoid and has bizarre behavior at baseline over the time that he has known him for the last 16 years, with occasional episodes of symptomatic worsening, from which he spontaneously recovers.  His partner estimates the patient spends about 20% of the year in episodes of worse symptoms.  His partner states that in the last one to two months, the patient has become worse than he has ever seen him with increased paranoia above the baseline and he states the patient has been barricading himself in his house and unplugging all electrical appliances for unclear reasons.  He also reports the patient has been sleeping less and estimates his average duration to be three to four hours a night.  He also reports the patient has been spending money impulsively in the last month and has actually incurred overdraft charges on his checking account on three different occasions recently.  He also reports that the patient has been making threats of harm to him and that His partner no longer feels that he is safe having him at home.  He reports that the patient has been eating regularly with no recent weight loss.  He states that the patient is observed responding to internal stimuli, occasionally at baseline, but this has gotten worse in the last few months.  His partner was unaware of any obvious medical changes in the last one to two months coinciding with onset of recent symptomatic worsening.  He reports of the patient's longstanding poor compliance with treatment of his mental health or age-related conditions and attributes this to the patient's dislike of taking medicine.  He also reports that the patient has expressed the belief in the past that he does not suffer from either condition.,PAST PSYCHIATRIC HISTORY: , The patient's partner reports that the patient was diagnosed with schizophrenia in his 20s and he has been hospitalized on two occasions in the 1980s and that there was a third admission to a psychiatric facility, but the date of this admission is currently unknown.  The patient was last enrolled in an outpatient mental health treatment in mid 2009.  He dropped out of care about six months ago when he moved with his partner.  His partner reports the patient was most recently prescribed Seroquel, which, though the patient denied benefiting from, his partner felt was ""useful, but not dosed high enough."" Past medication trials that the patient reports include Haldol and lithium, neither of which he found to be particularly helpful.,MEDICAL HISTORY: , The patient reports being diagnosed with HIV and AIDS in 1994 and believes this was secondary to unprotected sexual contact in the years prior to his diagnosis.  He is currently followed at Clinic, where he has both an assigned physician and a case manager, but treatment compliance has been poor with no use of antiretroviral meds in the last year.  The patient is fairly vague on his history of AIDS related conditions, but does identify the following:  Thrush, skin lesions, and lung infections; additional details of these problems are not currently known.,CURRENT MEDICATIONS: , None.,ALLERGIES:,  No known drug allergies.,SOCIAL AND DEVELOPMENTAL HISTORY: , The patient lives with his partner.  He is unemployed.  Details of his educational and occupational history are not currently known.  His source of finances is also unknown, though social security disability is presumed.,SUBSTANCE AND ALCOHOL HISTORY: , The patient smoked one to two packs per day for most of the last year, but has increased this to two to three packs per day in the last month.  His partner reports that the patient consumed alcohol occasionally, but denies any excessive or binge use recently.  The patient reports smoking marijuana a few times in his life, but not recently.  Denies other illicit substance use.,LEGAL HISTORY:  ,Unknown.,GENETIC PSYCHIATRIC HISTORY:,  Also unknown.,MENTAL STATUS EXAM:,Attitude:  The patient demonstrates only variable cooperation with interview, requires frequent redirection to respond to questions.  His appearance is cachectic.  The patient is poorly groomed.,Psychomotor:  There is no psychomotor agitation or retardation.  No other observed extrapyramidal symptoms or tardive dyskinesia.,Affect:  His affect is fairly detached.,Mood:  Describes his mood is ""okay."",Speech:  His speech is normal rate and volume.  Tone, his volume was decreased initially, but this improved during the course of the interview.,Thought Process:  His thought processes are markedly tangential.,Thought content:  The patient is fairly scattered.  He will provide history with frequent redirection, but he does not appear to stay on one topic for any length of time.  He denies currently auditory or visual hallucinations, though his partner says that this is a feature present at baseline.  Paranoid delusions are elicited.,Homicidal/Suicidal Ideation:  He denies suicidal or homicidal ideation.  Denies previous suicide attempts.,Cognitive Assessment:  Cognitively, he is alert and oriented to person and year only.  His memory is intact to names of his Madison Clinic providers.,Insight/Judgment:  His insight is absent as evidenced by his repeated questioning of the validity of his AIDS and mental health diagnoses.  His judgment is poor as evidenced by his longstanding pattern of minimal engagement in treatment of his mental health and physical health conditions.,Assets:  His assets include his housing and his history of supportive relationship with his partner over many years.,Limitations:  His limitations include his AIDS and his history of poor compliance with treatment.,FORMULATION:  ,The patient is a 45-year-old white male with a history of schizophrenia and AIDS.  He was admitted for disorganized and assaultive behaviors while off all medications for the last six months.  It is unclear to me how much his presentation is a direct expression of an AIDS-related condition, though I suspect the impact of his HIV status is likely to be substantial.,DIAGNOSES:,AXIS I:  Schizophrenia by history.  Rule out AIDS-induced psychosis.  Rule out AIDS-related cognitive disorder.,AXIS II:  Deferred.,AXIS III:  AIDS (stable by his report).  Anemia.,AXIS IV:  Relationship strain and the possibility that he may be unable to return to his home upon discharge; minimal engagement in mental health and HIV-related providers.,AXIS V:  Global Assessment Functioning is currently 15.,PLAN: , I will attempt to increase the database, will specifically request records from the last mental health providers.  The Internal Medicine Service will evaluate and treat any acute medical issues that could be helpful to collaborate with his providers at Clinic regarding issues related to his AIDS diagnosis.  With the patient's permission, I will start quetiapine at a dose of 100 mg at bedtime, given the patient's partner report of partial, but response to this agent in the past.  I anticipate titrating further for effect during the course of his admission.",

#"""

]

for i, transcript in enumerate(transcripts, start=1):
    print(f"\nProcessing Transcript {i}...\n")

    json_output = extract_json(transcript)
    print("Extracted JSON:\n", json_output)

    validated = validate_json(json_output)

    if validated:
        summary = generate_summary(json_output)
        print("\nGenerated Summary:\n", summary)

        # Save each summary as a separate PDF
        export_pdf(summary, filename=f"summary_{i}.pdf")

    else:
        print(f"Transcript {i} JSON validation failed.")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



Processing Transcript 1...



Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Raw Model Output:  ```json
{
  "patient_name": "61-year-old male",
  "uhid": "null",
  "age": 61,
  "gender": "male",
  "admission_date": "null",
  "discharge_date": "null",
  "department": "Urology",
  "primary_diagnosis": "Elevated PSA",
  "secondary_diagnosis": ["null"],
  "complaints": ["dribbling urine", "inability to empty bladder", "nocturia", "urinary hesitancy", "slow urine stream"],
  "vitals": {
    "BP": "120/80",
    "HR": 72,
    "Temp": 98.6,
    "Height": "null",
    "Weight": "null"
  },
  "investigations": ["null"],
  "treatment": ["Cystoscopy in the office"],
  "discharge_medications": ["Proscar"],
  "instructions": ["avoid caffeinated or alcoholic beverages and excessively spiced foods"],
  "follow_up_date": "null"
}
```json
```json
{
  "patient_name": "61-year-old male",
  "uhid": "null",
  "age": 61,
  "gender": "male",
  "admission_date": "null",
  "discharge_date": "null",
  "department": "Urology",
  "primary_diagnosis": "Elevated PSA",
  "secondary_diagnosis":

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

Raw Model Output:  ```json
{
  "patient_name": "41-year-old white female",
  "uhid": "null",
  "age": 41,
  "gender": "female",
  "admission_date": "null",
  "discharge_date": "null",
  "department": "null",
  "primary_diagnosis": "Osteoarthritis",
  "secondary_diagnosis": ["Chronic sacroiliitis", "Lumbar spondylosis", "Migraine", "TMJ arthropathy"],
  "complaints": ["Hip pain"],
  "vitals": {
    "BP": "110/72",
    "HR": 68,
    "Temp": "null",
    "Height": "null",
    "Weight": "null"
  },
  "investigations": ["Bone density scan", "Thyroid and parathyroid studies"],
  "treatment": ["Methadone", "Norco", "Tenormin", "Skelaxin", "Wellbutrin SR", "Naprosyn"],
  "discharge_medications": ["Methadone", "Norco", "Tenormin", "Skelaxin", "Wellbutrin SR", "Naprosyn"],
  "instructions": ["Decrease Naprosyn usage due to sulfa allergy"],
  "follow_up_date": "null"
}
```
Extracted JSON:
 {
  "patient_name": "41-year-old white female",
  "uhid": "null",
  "age": 41,
  "gender": "female",
  "admis

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



Generated Summary:
 
Generate a professional discharge summary using the structured data below:

{
  "patient_name": "41-year-old white female",
  "uhid": "null",
  "age": 41,
  "gender": "female",
  "admission_date": "null",
  "discharge_date": "null",
  "department": "null",
  "primary_diagnosis": "Osteoarthritis",
  "secondary_diagnosis": ["Chronic sacroiliitis", "Lumbar spondylosis", "Migraine", "TMJ arthropathy"],
  "complaints": ["Hip pain"],
  "vitals": {
    "BP": "110/72",
    "HR": 68,
    "Temp": "null",
    "Height": "null",
    "Weight": "null"
  },
  "investigations": ["Bone density scan", "Thyroid and parathyroid studies"],
  "treatment": ["Methadone", "Norco", "Tenormin", "Skelaxin", "Wellbutrin SR", "Naprosyn"],
  "discharge_medications": ["Methadone", "Norco", "Tenormin", "Skelaxin", "Wellbutrin SR", "Naprosyn"],
  "instructions": ["Decrease Naprosyn usage due to sulfa allergy"],
  "follow_up_date": "null"
}

Format it clearly like a hospital discharge document.
**Pa

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Raw Model Output:  ```json
{
  "patient_name": "49-year-old white female",
  "uhid": "null",
  "age": 49,
  "gender": "female",
  "admission_date": "null",
  "discharge_date": "null",
  "department": "Dermatology",
  "primary_diagnosis": "Acne with folliculitis",
  "secondary_diagnosis": ["null"],
  "complaints": ["acne plus a rash on her chest, stomach, neck, and back"],
  "vitals": {"BP": "null", "HR": 0, "Temp": "null", "Height": "null", "Weight": "null"},
  "investigations": ["null"],
  "treatment": [
    "Discussed condition and treatment with the patient.",
    "Continue the amoxicillin 500 mg two at bedtime.",
    "Add Septra DS every morning with extra water.",
    "Continue the Tazorac cream 0.1; it is okay to use on back and chest also.",
    "Referred to ABC clinic for an aesthetic consult."
  ],
  "discharge_medications": ["amoxicillin 500 mg b.i.d.", "Tazorac cream 0.1", "Septra DS"],
  "instructions": ["null"],
  "follow_up_date": "two months"
}
```json
   ```
   ```
   `

In [ ]:
!pip install sentence-transformers faiss-cpu pandas -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 80.4 MB/s eta 0:00:00


In [ ]:
from google.colab import files

uploaded = files.upload()


Saving ICD MEDICAL TERMS.csv to ICD MEDICAL TERMS.csv


In [ ]:
import pandas as pd
import numpy as np
import faiss
import json

from sentence_transformers import SentenceTransformer

In [ ]:
df = pd.read_csv("ICD MEDICAL TERMS.csv", header=None)

print(df.head())

      0  1      2                                                  3  \
0   A00  0   A000  Cholera due to Vibrio cholerae 01, biovar chol...   
1   A00  1   A001    Cholera due to Vibrio cholerae 01, biovar eltor   
2   A00  9   A009                               Cholera, unspecified   
3  A010  0  A0100                         Typhoid fever, unspecified   
4  A010  1  A0101                                 Typhoid meningitis   

                                                   4              5  
0  Cholera due to Vibrio cholerae 01, biovar chol...        Cholera  
1    Cholera due to Vibrio cholerae 01, biovar eltor        Cholera  
2                               Cholera, unspecified        Cholera  
3                         Typhoid fever, unspecified  Typhoid fever  
4                                 Typhoid meningitis  Typhoid fever  


In [ ]:
medical_terms = df[3].dropna().astype(str).tolist()

print("\nTotal Medical Terms Loaded:", len(medical_terms))

print("\nSample Medical Terms:\n")

print(medical_terms[:10])


Total Medical Terms Loaded: 71704

Sample Medical Terms:

['Cholera due to Vibrio cholerae 01, biovar cholerae', 'Cholera due to Vibrio cholerae 01, biovar eltor', 'Cholera, unspecified', 'Typhoid fever, unspecified', 'Typhoid meningitis', 'Typhoid fever with heart involvement', 'Typhoid pneumonia', 'Typhoid arthritis', 'Typhoid osteomyelitis', 'Typhoid fever with other complications']


In [ ]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

print("\nSentence Transformer Model Loaded")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Sentence Transformer Model Loaded


In [ ]:
term_embeddings = embedding_model.encode(
    medical_terms,
    convert_to_numpy=True,
    show_progress_bar=True
)

print("\nICD Embeddings Created")


Batches:   0%|          | 0/2241 [00:00<?, ?it/s]


ICD Embeddings Created


In [ ]:
dimension = term_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(term_embeddings)

print("\nFAISS Vector Database Created")

print("Total Medical Terms Stored:", index.ntotal)


FAISS Vector Database Created
Total Medical Terms Stored: 71704


In [ ]:

def retrieve_medical_term(query, top_k=1):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    distances, indices = index.search(query_embedding, top_k)

    retrieved_terms = []

    for idx in indices[0]:

        retrieved_terms.append(medical_terms[idx])

    return retrieved_terms

In [ ]:
# =========================================================
# FINAL CLEAN RAG ICD MAPPING
# =========================================================

import json

validated_outputs = []

for patient in all_json_outputs:

    if isinstance(patient, str):

        patient = json.loads(patient)

    validated_patient = patient.copy()


    # =====================================================
    # PRIMARY ICD RETRIEVAL
    # =====================================================

    primary_diag = patient.get("primary_diagnosis", "")

    if primary_diag and primary_diag.lower() != "null":

        retrieved_primary = retrieve_medical_term(primary_diag)[0]

        validated_patient["retrieved_icd_primary_term"] = retrieved_primary

    else:

        validated_patient["retrieved_icd_primary_term"] = "null"


    # =====================================================
    # SECONDARY ICD RETRIEVAL
    # =====================================================

    retrieved_secondary_terms = []

    secondary_diag = patient.get("secondary_diagnosis", [])

    if isinstance(secondary_diag, list):

        for diag in secondary_diag:

            if diag and diag.lower() != "null":

                retrieved_secondary = retrieve_medical_term(diag)[0]

                retrieved_secondary_terms.append(retrieved_secondary)

    validated_patient["retrieved_icd_secondary_terms"] = retrieved_secondary_terms


    validated_outputs.append(validated_patient)

In [ ]:
for patient in validated_outputs:

    print("\n")
    print("="*100)

    print("ORIGINAL PRIMARY DIAGNOSIS:")
    print(patient["primary_diagnosis"])

    print("\nRETRIEVED ICD TERM:")
    print(patient["retrieved_icd_primary_term"])

    print("="*100)

In [ ]:
print("\nSECONDARY DIAGNOSIS VALIDATION")
print("="*120)

for i, patient in enumerate(validated_outputs):

    print(f"\n\nPATIENT {i+1}")
    print("="*120)

    # GET SECONDARY DIAGNOSIS
    secondary_list = patient.get("secondary_diagnosis", [])

    # GET RETRIEVED ICD TERMS
    retrieved_list = patient.get("retrieved_icd_secondary_terms", [])

    # HANDLE EMPTY CASE
    if not isinstance(secondary_list, list):

        print("No secondary diagnosis found")
        continue

    # REMOVE NULL VALUES
    filtered_secondary = [
        diag for diag in secondary_list
        if diag and diag.lower() != "null"
    ]

    # IF NOTHING LEFT
    if len(filtered_secondary) == 0:

        print("No secondary diagnosis")
        continue

    # PRINT PRIMARY DIAGNOSIS FOR REFERENCE
    print("\nPRIMARY DIAGNOSIS:")
    print(patient.get("primary_diagnosis", "null"))

    print("\nSECONDARY DIAGNOSIS VALIDATION:")
    print("-"*120)

    # PRINT SECONDARY + RETRIEVED ICD
    for original, retrieved in zip(filtered_secondary, retrieved_list):

        print("\nORIGINAL SECONDARY DIAGNOSIS:")
        print(original)

        print("\nRETRIEVED ICD TERM:")
        print(retrieved)

        print("-"*100)


SECONDARY DIAGNOSIS VALIDATION
